In [1]:
%load_ext autoreload
%autoreload 2

In [22]:
import pandas as pd
from sklearn.pipeline import Pipeline
from src.data.preproces_dataset import TextCleanTransformer, DenseTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

In [4]:
df = pd.read_csv("../data/raw/train.csv")

X, y = df['text'], df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
def eval_metrics(actual, pred):
    con_m = confusion_matrix(actual, pred)
    accuracy = accuracy_score(actual, pred)
    f1 = f1_score(actual, pred)
    return con_m, accuracy, f1


In [13]:
from sklearn.naive_bayes import GaussianNB
model =  ('model', GaussianNB())

In [17]:
from sklearn.ensemble import RandomForestClassifier
model_params = {'n_estimators': 500,
                'criterion': 'entropy',
                'max_depth': 10,
                'n_jobs': 4,
                }

model =  ('model', RandomForestClassifier(**model_params))

In [41]:
from sklearn.svm import SVC
model_params = {'gamma': 2,
                'C': 5.5,
                'kernel': 'rbf',
                'class_weight': 'balanced',
                }

model =  ('model', SVC(**model_params))

In [37]:
from sklearn.neighbors import KNeighborsClassifier
model_params = {'n_neighbors': 15}

model =  ('model', KNeighborsClassifier(**model_params))

In [42]:
tfidf_params = {'ngram_range': (1, 5),
                'max_features': 4000}

pipe = Pipeline([('clean', TextCleanTransformer()),
                 ('tfidf', TfidfVectorizer(**tfidf_params)),
                 ('dense', DenseTransformer()),
                 # ('norm', Normalizer()),
                 ('stand', StandardScaler()),
                 model])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

cm, a, f = eval_metrics(y_test, y_pred)

print(cm)
print(f"ACCURACY: {a} \tF1 SCORE: {f}")

[[848  26]
 [511 138]]
ACCURACY: 0.6474064346684176 	F1 SCORE: 0.33948339483394835
